In [2]:
import torch
import torchvision # --> Contains MNIST data whuch we are going to use
import torch.nn as nn
import torch.optim as optim
import torchmetrics
import torchvision.transforms as transforms
import torchvision.transforms.v2 as T # --> Pytorch's CV library for dealing with images(resize, cropping , reshaping, pre-trained models CV)
from torch.utils.data import TensorDataset ,DataLoader # --> TensorDataset is for wrappimg dataset into a Tensor while DataLOader implemets Minibatch GD by loading data in batches
print('code run above successfully')

code run above successfully


## Build a neural netwwork with pytorch for classifying letters to read words

Data ingestion and data prep

In [3]:
## Normalization of data and preprocessing
transform = transforms.Compose([T.ToImage(),T.ToDtype(torch.float32 , scale = True)]) # --> T.ToTensor is deprecated do preferred to use ToImage

## Data is from MNIST database : contains thousands of images of handwritten didgits
train_data_and_validation_data = torchvision.datasets.MNIST(
    root = 'train_data_and_validation_data' , train = True , download = True ,  transform = transform
)
test_data = torchvision.datasets.MNIST(root = 'test_data' , download = True , train = False , transform = transform)

## Ensure results are reproducible
torch.manual_seed(42)

## Split train into train and validation
train_data , validation_data = torch.utils.random_split( train_data_and_validation_data , [55_000 , 5_000 ])

## Load data using data loader
train_data_loader = DataLoader(train_data , batch_size = 110 , shuffle = True)
validation_data_loader = DataLoader( validation_data , batch_size = 300 , shuffle  =  False)
test_data_laoder = DataLoader(test_data , batch_size = 500 , shuffle = False)

## Device management
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Currently using {device}")

100.0%


OSError: [Errno 28] No space left on device

Model building

In [ ]:
## Create a custom mutli-layer perceptron using a Wide and deep NN architecture
class MNIST_CLASSIFIER(nn.module):
    def __init__(self ,  n_inputs ,  n1 , n2 , n3):
        super().__init__()
        self.flatten = nn.Flatten()
        self.mlp = nn.Sequential(n_inputs , nn.ReLU(),
                                 nn.Linear(n1 , n2),nn.ReLU(),
                                 nn.Linear(n2 ,  n3),nn.ReLU())
        self.output = nn.Linear(n3 + n_inputs , 10 )

    def forward(self , x):
        x = self.flatten(x)
        deep_output = self.mlp(x)
        wide_and_deep = torch.concat([deep_output, x] , dim = 1) # --> While I was doing this I asked myself why it's torch and not tensor
        return self.output(wide_and_deep)
    
model = MNIST_CLASSIFIER(n_inputs = 784 ,  n1 = 418 , n2 = 128 , n3 = 64).to(device)
    
optimizer = optim.Adam(model.parameters(), lr = 1e-4)
loss_function = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task = 'multiclass' , num_classes = 10).to(device)
## Confirm model's parametres
print(next(model.parameters()).device)

Training

In [ ]:
def train_data( model , train_data , optimizer , device , loss_function ):
    model.train() # --> sets model to training mode
    loss_accumlation = 0 # --> Measusre loss
    print_interval = 100

    for batch_index , (data , target) in enumerate(data , 1):
        ## Ensure data is in same device
        data = data.to(device)
        ## Ensure target is in same device
        targets = targets.to(device)
        # foward prop
        y_pred = model(data)
        # calculate the loss
        loss = loss_function(y_pred , target)
        # backprop -- > the loss rep the loss function
        loss.backward()
        # Updates the parameters
        optimizer.step()
        # reset optimizers gradients
        optimizer.zero_grad()
        # trach progress 
        loss_accumlation +=loss.item()
        #Print 5 batches : Our data has 500 indices but we want to print the which provide this result batch_index % 100 == 0
        if batch_index % print_interval == 0 and batch_index < 0 :  # mean loss denominator is divided by print interval because it is the batch number , example we set print to be done after 100 batches meaning for 1 to 100 loss has been accumulating so average is just by the number of times it is accumulating
            # Get the loss for training the model 
            mean_loss = loss_accumlation / print_interval
            print(f"[{batch_index * 110} /{55_000}]"
            f"\nLoss : {mean_loss : .4f}")
            loss_accumlation = 0

Evaluation

In [ ]:
def evaluation(model ,  device , data , metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for inputs , targets in data:
            inputs =  inputs.to(device)
            targets = targets.to(device)
            y_pred = model(inputs)
            metric.update(y_pred , targets)
    return metric.compute()
            

Training loss

In [ ]:
for epoch in range(10):
    print(f"\n--- Epoch {epoch + 1} ---")
    train_data(model , train_data_loader , optimizer , device , loss_function)
    accuracy = evaluation(model , device , validation_data_loader , accuracy)
    print(f"Test Accuracy : {accuracy : .2f}")

inferencing